In [5]:
import os
import pandas as pd
from sqlalchemy import create_engine
from dotenv import load_dotenv

load_dotenv(override=True)

engine = create_engine(
    f"postgresql+psycopg2://{os.environ['DB_USER']}:{os.environ['DB_PASSWORD']}"
    f"@{os.environ['DB_HOST']}:{os.environ['DB_PORT']}/{os.environ['DB_NAME']}"
)

# sanity check
pd.read_sql("SELECT 1 AS ok", engine)

,ok
0,1


In [6]:
df = pd.read_sql("""
    SELECT ticker, title, category, ats
    FROM hiring_signals
    WHERE snapshot_date = (SELECT MAX(snapshot_date) FROM hiring_signals)
""", engine)

print(df.shape, "|", df['ticker'].nunique(), "tickers")
df.head()

(11020, 4) | 9 tickers


,ticker,title,category,ats
0,MU,Circuit Design Engineer,DTPG,eightfold
1,MU,Process Engineer,Front End,eightfold
2,MU,TECHNOLOGIST - PIE Yield Enhancement,Front End,eightfold
3,NVDA,Senior Deep Learning Framework Communications ...,Engineering,workday
4,NVDA,Principal Software Engineer - Rack Scale Syste...,Engineering,workday


Latest snapshot ^

In [7]:
from collections import Counter
import re

# Single-word frequency — the raw material for keyword rules
tokens = Counter()
for t in df['title'].dropna():
    tokens.update(re.findall(r'[a-z]+', t.lower()))

print("Top 40 title words:")
for word, n in tokens.most_common(40):
    print(f"{n:6d}  {word}")

# Top exact titles — catches multi-word role signals and shows the landscape
print("\nTop 25 exact titles:")
print(df['title'].value_counts().head(25).to_string())

Top 40 title words:
  7188  engineer
  3054  senior
  1668  design
  1384  software
  1320  staff
  1162  manager
   808  principal
   797  ai
   722  and
   689  product
   684  development
   676  sr
   666  verification
   654  architect
   588  engineering
   553  lead
   518  system
   449  process
   368  systems
   367  program
   333  test
   332  validation
   331  intern
   305  data
   282  solutions
   277  technical
   275  technician
   264  physical
   260  gpu
   254  hbm
   249  analog
   248  cpu
   240  performance
   236  hardware
   229  equipment
   226  firmware
   214  integration
   210  asic
   208  soc
   206  silicon

Top 25 exact titles:
title
Principal Design Engineer                            30
Application Engineer                                 26
Lead Software Engineer                               24
Design Verification Engineer                         23
Lead Design Engineer                                 22
Software Engineer II                   

Finds key words within titles and ranks them in frequency to be given to AI to create categories & category matches.

Next step, create a keyword classifier using AI to rapidly iterate in sprints, quickly classifying which key words have/haven't been captured

In [17]:
import re
from IPython.display import display

ROLE_RULES = [
    ("Product Management",        r"\b(product manager|product management|product owner|head of product|director of product)\b"),
    ("Program / Project Mgmt",    r"\b(program manager|project manager|technical program|tpm|scrum master|release manager|agile)\b"),
    ("Sales / Field / Marketing", r"\b(application engineer|applications engineer|field application|solution architect|solutions architect|solutions engineer|sales|account manager|account executive|account director|business development|customer success|field engineer|presales|marketing|developer relations|client manager|enablement)\b"),
    ("Data & Analytics",          r"\b(data scientist|data engineer|data analyst|analytics|business intelligence)\b"),
    ("Corporate / Ops / G&A",     r"\b(accountant|accounting|finance|financial|controller|human resources|hr|employee relations|recruit|talent acquisition|legal|counsel|paralegal|facilities|supply chain|supply management|procurement|sourcing|supplier management|supplier manager|buyer|purchasing|payroll|logistics|operations|cost control|business analyst|program analyst|it support|help desk|administrative|executive assistant|corporate communications)\b"),
    ("Research / Science",        r"\b(research scientist|researcher|scientist|applied scientist|principal scientist|research engineer|post-doc|postdoc|post doc)\b"),
    ("Verification & Validation", r"\b(verification|validation|formal verification|emulation)\b"),
    ("Software & Firmware",       r"\b(software|firmware|embedded|device driver|sdk|kernel|compiler|full stack|fullstack|developer|devops|site reliability)\b"),
    ("Manufacturing & Process",   r"\b(process|equipment|manufacturing|fab|hvm|yield|packaging|assembly|technician|reliability|failure analysis|test engineer|production|industrial|quality|supplier quality|mechanical|mfg|product engineer|product development|module|npi|amhs)\b"),
    ("Design (silicon/IC)",       r"\b(design|designer|analog|mixed signal|mixed-signal|rtl|layout|asic|soc|silicon|chip|dft|vlsi|circuit|standard cell|cad)\b"),
    ("Systems & Architecture",    r"\b(systems|system engineer|architect|architecture|integration|signal integrity|performance|hardware|board)\b"),
    ("Engineering (unspecified)", r"\b(engineer|engineering)\b"),
    # NEW — parallel to Engineering (unspecified): management roles naming no function
    ("Management (general)",      r"\b(manager|mgr|director|management|head of|vice president|vp|chief|supervisor)\b"),
]
ROLE_RULES = [(name, re.compile(pat)) for name, pat in ROLE_RULES]

def classify_role(title):
    t = title.lower()
    for name, pat in ROLE_RULES:
        if pat.search(t):
            return name
    return "Other / Unclassified"

df["role_bucket"] = df["title"].fillna("").apply(classify_role)
df["is_ai"]       = df["title"].fillna("").apply(lambda t: bool(AI_PAT.search(t.lower())))

print(f"Unclassified: {(df['role_bucket']=='Other / Unclassified').mean():.1%}   |   "
      f"AI-flagged: {df['is_ai'].mean():.1%} ({int(df['is_ai'].sum())} jobs)")
print("\nRole bucket distribution:")
display(df["role_bucket"].value_counts())
print("\nWhat's left in Other:")
display(df.loc[df['role_bucket']=='Other / Unclassified', 'title'].value_counts().head(15))

Unclassified: 4.4%   |   AI-flagged: 9.4% (1037 jobs)

Role bucket distribution:


role_bucket
Manufacturing & Process      1802
Design (silicon/IC)          1707
Software & Firmware          1648
Engineering (unspecified)    1392
Verification & Validation    1005
Sales / Field / Marketing     831
Systems & Architecture        699
Corporate / Ops / G&A         603
Other / Unclassified          480
Program / Project Mgmt        304
Management (general)          232
Product Management            109
Research / Science            107
Data & Analytics              101
Name: count, dtype: int64


What's left in Other:


title
Technical specialist, OMT RDA-2 Operation_TC               4
Cyber Security Analyst                                     2
Machine Learning Intern - 2026                             2
Data Center Deployment Specialist                          2
Automotive SW PM                                           2
Medium Voltage Distribution Electrician                    2
Senior Consultant - Datacom                                2
Intern - MMP pFA Lab                                       2
FE CPIE PI MTS                                             2
Member of Technical Staff (MTS), Machine Learning, SMAI    2
SR ENGR, FE CENTRAL QE Product                             2
【苗栗銅鑼】製程安全工程師                                              2
TPG Workforce Development-Onboarding                       2
TECHNOLOGIST - CHEMLAB                                     2
ASSOC. ENG, OMT MQC LAB/CC_TY                              2
Name: count, dtype: int64

In [20]:
mix = pd.crosstab(df['ticker'], df['role_bucket'], normalize='index').mul(100).round(1)
mix = mix[df['role_bucket'].value_counts().index]      # columns ordered by overall size

print("Jobs per ticker (N):")
print(df['ticker'].value_counts().sort_index().to_string())

print("\nRole mix by ticker (% of each ticker's roles):")
display(mix.T)   # buckets as rows, tickers as columns — easy to compare across a row

Jobs per ticker (N):
ticker
AMD     1063
AVGO     342
CDNS     641
INTC     651
MRVL     713
MU      2991
NVDA    2611
QCOM    1536
TXN      472

Role mix by ticker (% of each ticker's roles):


ticker,AMD,AVGO,CDNS,INTC,MRVL,MU,NVDA,QCOM,TXN
role_bucket,,,,,,,,,
Manufacturing & Process,9.7,21.1,8.7,18.9,6.3,34.4,7.4,4.9,22.2
Design (silicon/IC),19.2,23.4,17.5,21.2,34.8,10.6,11.5,14.3,18.6
Software & Firmware,18.1,16.7,22.5,9.8,6.6,2.5,26.6,22.6,5.9
Engineering (unspecified),6.0,5.6,6.7,8.3,5.9,19.6,8.5,20.7,9.3
Verification & Validation,14.3,5.3,11.9,13.1,24.5,3.9,9.1,6.4,9.5
Sales / Field / Marketing,6.5,5.8,20.1,6.0,4.3,2.0,13.2,4.8,13.8
Systems & Architecture,8.8,7.6,4.4,4.8,4.9,3.0,8.2,10.4,4.4
Corporate / Ops / G&A,4.9,3.5,3.1,7.7,3.8,9.5,2.5,4.1,6.4
Other / Unclassified,2.9,4.4,3.3,4.3,2.5,7.1,2.0,4.4,7.6
